In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from sliderule import icesat2

### Configuration

In [ ]:
atl24_granule = "s3://sliderule-public/atl24r3/parquet/ATL24_20181028071900_04530107_006_02_003_01.parquet"
selected_ground_track = icesat2.GT1L

### Read Input File

In [ ]:
gdf = gpd.read_parquet(atl24_granule)
gt = gdf[gdf["gt"] == selected_ground_track]

### Display Contents of GeoDataFrame

In [ ]:
gt

### Plot Photon Classifications

In [ ]:
# class_ph label + color mapping
class_labels = {
    0:  ("unclassified", "#999999"),
    1:  ("other",        "#9467bd"),
    2:  ("ground",       "#2ca02c"),
    40: ("bathymetry",   "#d62728"),
    41: ("sea surface",  "#1f77b4"),
}

fig, ax = plt.subplots(figsize=(14, 6))

for cval, (label, color) in class_labels.items():
    sub = gt[gt["class_ph"] == cval]
    if len(sub) == 0:
        continue
    ax.scatter(sub["x_atc"], sub["geoid_corr_h"],
               s=2, c=color, label=f"{cval}: {label}")

ax.set_xlabel("x_atc (m)")
ax.set_ylabel("ortho_h (m)")
ax.set_title("GT1L — ortho_h vs x_atc colored by class_ph")
ax.legend(markerscale=4, loc="best")
plt.tight_layout()
plt.show()

### Plot Uncertainties

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# sigma_tvu is a positive uncertainty magnitude; scale the colormap to a
# robust range of the actual data so the variation is visible
vmin = np.nanpercentile(gt["sigma_tvu"], 1)
vmax = np.nanpercentile(gt["sigma_tvu"], 99)

sc = ax.scatter(gt["x_atc"], gt["geoid_corr_h"],
                s=2, c=gt["sigma_tvu"],
                cmap="viridis", vmin=vmin, vmax=vmax)

cbar = fig.colorbar(sc, ax=ax, extend="both")
cbar.set_label("sigma_tvu (m)")

ax.set_xlabel("x_atc (m)")
ax.set_ylabel("ortho_h (m)")
ax.set_title("GT1L — ortho_h vs x_atc colored by sigma_tvu")
plt.tight_layout()
plt.show()